# Chess — ThompsonZero **FULL**

A revamp of the ThompsonZero methodology. The network has a common ResNet trunk
and **two learned heads**; two more heads are **derived** (closed-form, no extra
parameters) from them:

| head | output | meaning |
|---|---|---|
| **policy-prior** (learned) | per-action logits + scalar `β` | `π_a = softmax(logits)` — a Dirichlet-categorical belief over *which move is optimal*, prior strength `β` |
| **action-value** (learned) | per action: `(p_win,p_draw,p_loss)` + conc. `c_a` | `pv_a = Dir(c_a·(p_w,p_d,p_l))` — belief over that action's 3-outcome result |
| **value** (derived) | scalar `v` | `v = Σ_a π_a (p_win_a − p_loss_a)` |
| **value-dist** (derived) | Dirichlet `qv` | moment-match of the π-weighted mixture `Σ_a π_a·pv_a` (a mixture of Dirichlets → one Dirichlet by matching its exact mean+variance) |

### Search — Thompson sampling over the per-action value beliefs
Expanding a node runs **one** NN forward pass (value + policy). Selection at
every node **Thompson-samples each edge's `pv_a`** (one `(w,d,l)` draw → value
`v=w−l`) and descends into the **argmax** — no PUCT constant, no exploration
noise (posterior sampling explores on its own). `π_a` is **not** used to steer
selection. A `temperature` **multiplies the concentrations** before sampling
(`T=1` as-is early; `T>1` sharpens toward the mean ≈ max for the endgame).

Backup, bottom-up along the path: **(a)** credit the selected edge with one
policy-prior observation (`pcount += 1`) so `π_a` converges to the visit
distribution and `qv`'s mixture weights sharpen onto the best move; **(b)**
recompute the node's `qv` as the π-weighted mixture; **(c)** **replace** the
parent's `pv_a` with the child's `qv`, **flipped** one ply (`win↔loss`). Nothing
is accumulated for the value — the belief is *recomputed* from the child, so a
fresh leaf estimate propagates to the root. **MCTS-Solver** overlays exact
win/draw/loss proofs (`WIN>DRAW>LOSS`); a proven **win** also gets extra policy
pseudocounts (`PROVEN_WIN_BONUS`) so the prior learns forced mates fast.

### Training — four losses summed
`L = w_pol·L_policyprior + w_av·L_actionvaluedist + w_vd·L_valuedist + w_v·L_value`

- **`L_policyprior`** — `KL( Dir(visit counts) ‖ Dir(β·π) )`, closed form. (This
  supervises `β` too; if the concentration proves inert it can later collapse to
  a plain categorical + cross-entropy.)
- **`L_actionvaluedist`** — `KL( Dir(backed-up pv_a) ‖ Dir(predicted pv_a) )` on
  **evidence edges only** (expanded/proven), closed form.
- **`L_valuedist`** — `KL( Dir(backed-up qv) ‖ Dir(predicted qv) )`, closed form.
- **`L_value`** — MSE between the game outcome `z` and the derived `v` (AlphaZero
  value loss); `z` is mixed into the `qv` target too (γ-ramped `Z_MIX`).

All three KLs are forward `KL(target‖prediction)`. `lgamma`/`digamma` have no
DirectML kernel, so on that backend the loss is computed on CPU-gathered head
entries and its gradients are seeded back into the device graph (no exotic op
ever runs on the DML device).

### Sparse deep eval (fixed cost per checkpoint)
Every checkpoint enters ONE running-Elo table rated at **MCTS-128**, alongside a
`random` mover. Per new checkpoint: it plays a fixed number of games vs the
**last 3** checkpoints + `random`, then **10 random pairs** from the whole pool
play too (keeps old ratings mixing). Elo `K` **decays with the games a pair has
already played**, so ratings settle. Cost is independent of pool size.

### Efficiencies kept from the repo
Native C++ chess, batched-leaf waves shared across parallel self-play games
(one forward pass per wave), Thompson-diverse waves + virtual loss, MCTS-Solver,
tree reuse, playout-cap randomization, z-mix grounding, opponent-pool diversity,
backward-restart curriculum, DirectML-safe GroupNorm/softplus/AdamW, checkpoint
resume. (Chess has no board-symmetry augmentation.) The heavy lifting lives in
`chess_thompson_full_utils.py`; this notebook holds only the hyperparameters and
the driver.

> **Design notes / open levers.** `qv` is a *soft-average* (π-weighted) value —
> it approaches the max only as `π` sharpens (intended). `pv_a` is *replaced* by
> the child `qv` (both model the same thing; `qv` carries more info). The policy
> prior is not used in selection (v1). Repeated moment-matching up the tree loses
> correlation structure (accepted for closed-form KLs). See the PR description
> for the full rationale.

In [ ]:
%pip install open_spiel -q
# Load the shared implementation module. If this notebook is run from inside the
# repo the file sits next to it; on Colab we fetch it from the branch.
import os, sys, urllib.request
_UTILS = 'chess_thompson_full_utils.py'
_BRANCH = 'claude/thompson_full'
_URL = ('https://raw.githubusercontent.com/calvinpozderac-claude/open_spiel/'
        f'{_BRANCH}/open_spiel/colabs/{_UTILS}')
for _p in ('.', 'open_spiel/colabs', os.path.dirname(__file__) if '__file__' in dir() else '.'):
    if os.path.exists(os.path.join(_p, _UTILS)):
        sys.path.insert(0, _p); break
else:
    urllib.request.urlretrieve(_URL, _UTILS); sys.path.insert(0, '.')
import chess_thompson_full_utils as ctf
print('utils:', ctf.__file__)

In [ ]:
# ── Hyperparameters (the ONLY tunables; everything else is in the utils file) ──
# Run
NUM_EPISODES     = 50_000       # budget only — LR schedule is decoupled below
CHANNELS         = 128
NUM_BLOCKS       = 10

# Device / IO
DEVICE_PREFERENCE   = 'auto'    # 'cpu' | 'cuda' | 'directml' | 'auto'
CHECKPOINT_DIR      = 'chess_thompson_full_ckpt'
RESUME_FROM_CHECKPOINT = True

# Self-play (batched across games; one NN forward per leaf wave)
N_PARALLEL_GAMES = 8
WAVE_PER_GAME    = 8
FAST_MCTS_SIMS   = 250          # 75% of games
FULL_MCTS_SIMS   = 1000         # 25% of games
FAST_GAME_PROB   = 0.75
TEMP_THRESHOLD   = 20           # Thompson-sample the move for the first N plies,
LATE_TEMP        = 8.0          # then sharpen (conc ×LATE_TEMP ≈ greedy max)
CONF_CAP         = 100.0        # cap on any training-target concentration
PROVEN_WIN_BONUS = 4.0          # extra policy pseudocounts on a proven mate
MAX_SELFPLAY_PLIES = 400
Z_MIX            = 0.5          # blend the game outcome into the qv value target
Z_GAMMA          = 0.97         # end-of-game ramp on Z_MIX
SELFPLAY_POOL_PROB = 0.15       # frac. of games with one side a frozen benchmark
RANDOM_POOL_FRAC   = 0.5        # …of those, frac. played vs a uniform-random side
RESTART_PROB     = 0.30         # backward-restart curriculum
RESTART_K_MIN, RESTART_K_MAX, RESTART_POOL_CAP = 2, 30, 128

# Training
BATCH_SIZE       = 512
TRAIN_STEPS_PER_EP = 8
MAX_BUFFER       = 150_000
LR_PEAK          = 2e-3
LR_WARMUP_EPS    = 100
LR_DECAY_EPS     = 12_000       # cosine horizon (decoupled from NUM_EPISODES)
LR_MIN_FACTOR    = 0.10
WEIGHT_DECAY     = 1e-4
GRAD_CLIP        = 1.0
# (L_value, L_valuedist, L_actionvaluedist, L_policyprior) — start equal; the KL
# terms run larger than the MSE, so down-weight them here if they dominate.
LOSS_WEIGHTS     = (1.0, 1.0, 1.0, 1.0)

# Sparse deep eval (fixed cost per checkpoint)
EVAL_EVERY          = 500       # add a checkpoint + run its eval every N eps
EVAL_SIMS           = 128       # every rated player searches at MCTS-128
EVAL_GAMES_PER_PAIR = 6         # games per matched pair (alternating colours)
EVAL_LAST_N         = 3         # new checkpoint plays the last-3 + random
EVAL_REFRESH_PAIRS  = 10        # random pool pairs refreshed each checkpoint
EVAL_OPENING_PLIES  = 4         # random opening plies for variety
EVAL_K_BASE         = 32.0      # Elo K, decayed by games-per-pair
EVAL_K_HALFLIFE     = 30.0      # K = K_BASE·HL/(HL + games_this_pair)
EVAL_TEMP           = 6.0       # concentration multiplier for eval MCTS moves
START_ELO           = 1000.0

In [ ]:
import numpy as np, torch, random, pyspiel

DEVICE, BACKEND = ctf.pick_device(DEVICE_PREFERENCE)
game = pyspiel.load_game('chess')
ctf.set_game(game)
EVAL_DEVICE = 'cpu'             # batch-1 eval inference: CPU replica

base_network = ctf.ThompsonFullNet(CHANNELS, NUM_BLOCKS).to(DEVICE)
network = base_network
if BACKEND != 'directml' and hasattr(torch, 'compile'):
    import shutil
    if shutil.which('g++') or shutil.which('gcc') or shutil.which('clang') or shutil.which('cl'):
        try:
            _c = torch.compile(base_network, dynamic=True)
            with torch.no_grad(): _c(torch.zeros(1, *ctf._OBS_SHAPE, device=DEVICE))
            network = _c; print('torch.compile: enabled')
        except Exception as e:
            print(f'torch.compile: disabled ({type(e).__name__}) — eager')

optimizer = (ctf.LerpFreeAdamW if BACKEND == 'directml' else torch.optim.AdamW)(
    network.parameters(), lr=LR_PEAK, weight_decay=WEIGHT_DECAY)

def _lr(ep):
    if ep < LR_WARMUP_EPS: return ep / max(LR_WARMUP_EPS, 1)
    f = min((ep - LR_WARMUP_EPS) / max(LR_DECAY_EPS - LR_WARMUP_EPS, 1), 1.0)
    return LR_MIN_FACTOR + (1 - LR_MIN_FACTOR) * 0.5 * (1 + np.cos(np.pi * f))
scheduler = torch.optim.lr_scheduler.LambdaLR(optimizer, _lr)

self_play = ctf.ThompsonParallelSelfPlay(
    game, network, DEVICE, n_parallel=N_PARALLEL_GAMES, wave_per_game=WAVE_PER_GAME,
    fast_sims=FAST_MCTS_SIMS, full_sims=FULL_MCTS_SIMS, fast_prob=FAST_GAME_PROB,
    temp_threshold=TEMP_THRESHOLD, late_temp=LATE_TEMP, conf_cap=CONF_CAP,
    proven_win_bonus=PROVEN_WIN_BONUS, max_plies=MAX_SELFPLAY_PLIES,
    z_mix=Z_MIX, z_gamma=Z_GAMMA, pool_prob=SELFPLAY_POOL_PROB,
    checkpoint_dir=CHECKPOINT_DIR, channels=CHANNELS, num_blocks=NUM_BLOCKS,
    restart_prob=RESTART_PROB, restart_k_min=RESTART_K_MIN,
    restart_k_max=RESTART_K_MAX, restart_pool_cap=RESTART_POOL_CAP,
    random_pool_frac=RANDOM_POOL_FRAC, seed=0)
episode_stream = self_play.episodes()

elo_pool = ctf.EloPool(game, EVAL_DEVICE, eval_sims=EVAL_SIMS, k_base=EVAL_K_BASE,
    k_halflife=EVAL_K_HALFLIFE, games_per_pair=EVAL_GAMES_PER_PAIR,
    last_n=EVAL_LAST_N, refresh_pairs=EVAL_REFRESH_PAIRS,
    opening_plies=EVAL_OPENING_PLIES, batch_size=16, start_elo=START_ELO,
    eval_temp=EVAL_TEMP, seed=0)

hist = {'ep': [], 'loss': [], 'pol': [], 'av': [], 'vd': [], 'val': [],
        'draw_pct': [], 'plies': [], 'buf': [], 'aux': [], 'elo': []}
replay_buffer, start_ep, aux_total = [], 1, 0

ckpt = ctf.load_checkpoint(CHECKPOINT_DIR) if RESUME_FROM_CHECKPOINT else None
if ckpt is not None:
    base_network.load_state_dict(ckpt['model']); optimizer.load_state_dict(ckpt['optim'])
    if ckpt.get('sched'): scheduler.load_state_dict(ckpt['sched'])
    elo_pool.elo = ckpt['elo']; elo_pool.order = ckpt['order']
    elo_pool.pair_games = {frozenset(k): v for k, v in ckpt['pair_games'].items()}
    elo_pool.players = ['random'] + list(elo_pool.order)
    for lb in elo_pool.order:
        elo_pool.nets[lb] = ctf.ThompsonFullNet(CHANNELS, NUM_BLOCKS)
        elo_pool.nets[lb].load_state_dict(torch.load(
            f'{CHECKPOINT_DIR}/bench_{lb}.pt', map_location='cpu', weights_only=True))
        elo_pool.nets[lb].eval()
    hist = ckpt['hist']; start_ep = ckpt['ep'] + 1
    print(f'resumed at ep {ckpt["ep"]}')

n_params = sum(p.numel() for p in base_network.parameters())
print(f'device={DEVICE} backend={BACKEND} | params={n_params:,} | start_ep={start_ep}')

if start_ep == 1:                                   # seed the Elo pool at iter 0
    init = ctf.cpu_clone(base_network, CHANNELS, NUM_BLOCKS)
    ctf.save_benchmark_net(CHECKPOINT_DIR, '0', init)
    elo_pool.add_checkpoint('0', init)

_prev = dict(self_play.stats)
for ep in range(start_ep, NUM_EPISODES + 1):
    network.eval()
    raw = next(episode_stream)
    aux_total += self_play.last_aux
    replay_buffer.extend(raw)
    if len(replay_buffer) > MAX_BUFFER: del replay_buffer[:-MAX_BUFFER]

    network.train()
    accs = {'loss': [], 'pol': [], 'av': [], 'vd': [], 'val': []}
    if len(replay_buffer) >= BATCH_SIZE:
        for _ in range(TRAIN_STEPS_PER_EP):
            batch = random.sample(replay_buffer, BATCH_SIZE)
            lv, parts = ctf.train_step(network, optimizer, batch, DEVICE,
                                       BACKEND, LOSS_WEIGHTS, GRAD_CLIP)
            accs['loss'].append(lv)
            for k in ('pol', 'av', 'vd', 'val'): accs[k].append(parts[k])
        scheduler.step()

    if ep % EVAL_EVERY != 0:
        continue
    snap = ctf.cpu_clone(base_network, CHANNELS, NUM_BLOCKS)
    ctf.save_benchmark_net(CHECKPOINT_DIR, str(ep), snap)
    elo = elo_pool.add_checkpoint(str(ep), snap)
    ctf.save_checkpoint(CHECKPOINT_DIR, ep, base_network, optimizer, scheduler,
                        elo_pool, hist)
    st = self_play.stats
    dg = max(st['games'] - _prev['games'], 1)
    draw_pct = 100 * (st['draw'] - _prev['draw']) / dg
    plies = (st['plies'] - _prev['plies']) / dg
    _prev = dict(st)
    ml = lambda k: float(np.mean(accs[k])) if accs[k] else float('nan')
    hist['ep'].append(ep); hist['loss'].append(ml('loss'))
    for k in ('pol', 'av', 'vd', 'val'): hist[k].append(ml(k))
    hist['draw_pct'].append(draw_pct); hist['plies'].append(plies)
    hist['buf'].append(len(replay_buffer)); hist['aux'].append(aux_total)
    hist['elo'].append(dict(elo))
    ladder = '  '.join(f'{k}={v:.0f}' for k, v in
                       sorted(elo.items(), key=lambda kv: -kv[1])[:6])
    print(f'ep {ep:6d} | loss {ml("loss"):.2f} '
          f'(pol {ml("pol"):.2f} av {ml("av"):.2f} vd {ml("vd"):.2f} val {ml("val"):.3f}) '
          f'| dr {draw_pct:.0f}% ply {plies:.0f} buf {len(replay_buffer)//1000}k '
          f'aux {aux_total} | lr {optimizer.param_groups[0]["lr"]:.2e}')
    print(f'         Elo@{EVAL_SIMS}: {ladder}')

In [ ]:
import matplotlib.pyplot as plt
fig, ax = plt.subplots(2, 3, figsize=(17, 8)); fig.suptitle('ThompsonZero-FULL (chess)')
ax[0,0].plot(hist['ep'], hist['pol'], label='policy KL')
ax[0,0].plot(hist['ep'], hist['av'],  label='action-value KL')
ax[0,0].plot(hist['ep'], hist['vd'],  label='value-dist KL')
ax[0,0].set_title('Distributional losses'); ax[0,0].legend(fontsize=8); ax[0,0].set_xlabel('ep')
ax[0,1].plot(hist['ep'], hist['val'], color='tab:orange'); ax[0,1].set_title('Value MSE (→ z)')
ax[0,2].axhline(START_ELO, color='gray', ls='--', lw=.8)
for lb in (hist['elo'][-1].keys() if hist['elo'] else []):
    xs = [e for e, s in zip(hist['ep'], hist['elo']) if lb in s]
    ys = [s[lb] for s in hist['elo'] if lb in s]
    if lb != 'random': ax[0,2].plot(xs, ys, marker='.', label=lb)
ax[0,2].set_title(f'Checkpoint Elo @ MCTS-{EVAL_SIMS}'); ax[0,2].set_xlabel('ep')
ax[1,0].plot(hist['ep'], hist['draw_pct'], color='gray'); ax[1,0].set_title('Self-play draw %')
ax[1,1].plot(hist['ep'], hist['plies'], color='tab:green'); ax[1,1].set_title('Mean game length (plies)')
ax[1,2].plot(hist['ep'], hist['buf'], label='buffer'); ax[1,2].plot(hist['ep'], hist['aux'], label='aux (solver)')
ax[1,2].set_title('Buffer / solver-labelled'); ax[1,2].legend(fontsize=8)
plt.tight_layout(); plt.show()

## Arena — pit two checkpoints (or policy vs its own search)

Load any two saved benchmarks and play them head-to-head at a chosen search
budget, or measure **how much search adds** by pitting a checkpoint's raw policy
against its own MCTS-128.

In [ ]:
import numpy as np
def _load(label):
    net = ctf.ThompsonFullNet(CHANNELS, NUM_BLOCKS)
    net.load_state_dict(torch.load(f'{CHECKPOINT_DIR}/bench_{label}.pt',
                                   map_location='cpu', weights_only=True))
    net.eval(); return net

def duel(label_a, label_b, sims_a=128, sims_b=128, n_games=20, seed=0):
    # label_* may be a checkpoint label or 'random'. sims=0 -> raw policy head.
    rng = np.random.RandomState(seed); wa = d = wb = 0
    def mk(lb, sims):
        if lb == 'random': return None
        net = _load(lb)
        return ('policy', net) if sims == 0 else (
            'mcts', ctf.ThompsonMCTSBot(game, net, 'cpu', sims, temp=EVAL_TEMP,
                                        random_state=rng))
    A, B = mk(label_a, sims_a), mk(label_b, sims_b)
    for g in range(n_games):
        players = (A, B) if g % 2 == 0 else (B, A)
        state = game.new_initial_state()
        while not state.is_terminal():
            p = players[state.current_player()]
            if p is None:
                leg = state.legal_actions(); a = int(leg[rng.randint(len(leg))])
            elif p[0] == 'policy':
                a = ctf.policy_move(p[1], state, 'cpu')
            else:
                a = ctf.root_pick(p[1].mcts_search(state), rng, thompson=False)
            state.apply_action(a)
        r = state.returns()[0 if g % 2 == 0 else 1]   # from label_a's view
        wa += r > 0; wb += r < 0; d += r == 0
    print(f'{label_a}(@{sims_a}) vs {label_b}(@{sims_b}): '
          f'W{wa} D{d} L{wb} over {n_games}')
    return wa, d, wb

# Examples (uncomment once you have checkpoints):
# duel('2000', '1000', 128, 128, 20)          # generation vs generation
# duel('2000', '2000', 0, 128, 20)            # raw policy vs its own search
# duel('2000', 'random', 128, 0, 20)